In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install einops
%cd /content/gdrive/MyDrive
!ls

import argparse
from concurrent.futures import process
import os
import sys
import datetime
import time
import math
import json
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torchvision import models as torchvision_models
from utils import utils_ssl as utils
from utils import projection_head
from functools import partial
from einops import rearrange, repeat
from einops.layers.torch import Rearrange





     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.3 MB/s eta 0:00:00
/content/gdrive/MyDrive
 checkPoint-Major-Poject      checkpoints	 models
 checkPoint-Major-Poject-v3  'Colab Notebooks'	 utils


In [ ]:
!pwd
!ls

from models.vit import VisionTransformer


/content/gdrive/MyDrive
 checkPoint-Major-Poject      checkpoints	 models
 checkPoint-Major-Poject-v3  'Colab Notebooks'	 utils


In [ ]:
class DataAugmentation(object):
    def __init__(self, args):
        if args["dataset"]=="CIFAR10":
            mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        elif args["dataset"] == "CIFAR100":
            mean, std = (0.5070, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)
        elif args["dataset"] == "SVHN":
            mean, std = (0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970)
        elif args["dataset"] == "CINIC":
            mean, std = (0.47889522, 0.47227842, 0.43047404),(0.24205776, 0.23828046, 0.25874835)
        elif args["dataset"] == "Tiny-Imagenet":
            mean, std = (0.4802, 0.4481, 0.3975), (0.2770, 0.2691, 0.2821)


        flip_and_color_jitter = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply(
                [transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)],
                p=0.8
            ),
            transforms.RandomGrayscale(p=0.2),
        ])
        normalize = transforms.Compose([
            transforms.ToTensor(),
	    transforms.Normalize(mean, std),
        ])

        # first global crop
        self.global_transfo1 = transforms.Compose([
            transforms.RandomResizedCrop(args["image_size"], scale=args["global_crops_scale"], interpolation=Image.BICUBIC),
            flip_and_color_jitter,
            utils.GaussianBlur(1.0),
            normalize,
        ])
        # second global crop
        self.global_transfo2 = transforms.Compose([
            transforms.RandomResizedCrop(args["image_size"], scale=args["global_crops_scale"], interpolation=Image.BICUBIC),
            flip_and_color_jitter,
            utils.GaussianBlur(0.1),
            utils.Solarization(0.2),
            normalize,
        ])
        # transformation for the local small crops
        self.local_crops_number = args["local_crops_number"]
        self.local_transfo = transforms.Compose([
            transforms.RandomResizedCrop(args["image_size"]//2, scale=args["local_crops_scale"], interpolation=Image.BICUBIC),
            flip_and_color_jitter,
            utils.GaussianBlur(p=0.5),
            normalize,
        ])

    def __call__(self, image):
        crops = []
        crops.append(self.global_transfo1(image))
        crops.append(self.global_transfo2(image))
        for _ in range(self.local_crops_number):
            crops.append(self.local_transfo(image))
        return crops

def setup_for_distributed(is_master):
    """
    This function disables printing when not in master process
    """
    import builtins as __builtin__
    builtin_print = __builtin__.print

    def print(*args, **kwargs):
        force = kwargs.pop('force', False)
        if is_master or force:
            builtin_print(*args, **kwargs)

    __builtin__.print = print

def init_distributed_mode(args):
    # launched with torch.distributed.launch
    if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ:
        args["rank"] = int(os.environ["RANK"])
        args["world_size"] = int(os.environ['WORLD_SIZE'])
        args["gpu"] = int(os.environ['LOCAL_RANK'])
    # launched with submitit on a slurm cluster
    elif 'SLURM_PROCID' in os.environ:
        args["rank"] = int(os.environ['SLURM_PROCID'])
        args["gpu"] = args["rank"] % torch.cuda.device_count()
    # launched naively with `python main_dino.py`
    # we manually add MASTER_ADDR and MASTER_PORT to env variables
    elif torch.cuda.is_available():
        print('Will run the code on one GPU.')
        args["rank"], args["gpu"], args["world_size"] = 0, 0, 1
        os.environ['MASTER_ADDR'] = '127.0.0.1'
        os.environ['MASTER_PORT'] = '29500'
    else:
        print('Does not support training without GPU.')
        sys.exit(1)

    dist.init_process_group(
        backend="nccl",
        init_method=args["dist_url"],
        world_size=args["world_size"],
        rank=args["rank"],
    )

    torch.cuda.set_device(args["gpu"])
    print('| distributed init (rank {}): {}'.format(
        args["rank"], args["dist_url"]), flush=True)
    dist.barrier()
    setup_for_distributed(args["rank"] == 0)

In [ ]:
class ViewPredLoss(nn.Module):
    def __init__(self, out_dim, ncrops, warmup_teacher_temp, teacher_temp,
                 warmup_teacher_temp_epochs, nepochs, in_channels=3, student_temp=0.1,
                 center_momentum=0.9):
        super().__init__()
        self.student_temp = student_temp
        self.center_momentum = center_momentum
        self.ncrops = ncrops
        self.in_channels = in_channels
        self.register_buffer("center", torch.zeros(1, out_dim))
        self.teacher_temp_schedule = np.concatenate((
            np.linspace(warmup_teacher_temp,
                        teacher_temp, warmup_teacher_temp_epochs),
            np.ones(nepochs - warmup_teacher_temp_epochs) * teacher_temp
        ))

    def forward(self, student_output, teacher_output, epoch):
        """
        Cross-entropy between softmax outputs of the teacher and student networks.
        """
        student_out = student_output / self.student_temp
        student_out = student_out.chunk(self.ncrops)


        # teacher centering and sharpening
        temp = self.teacher_temp_schedule[epoch]
        teacher_out = F.softmax((teacher_output - self.center) / temp, dim=-1)
        teacher_out = teacher_out.detach().chunk(2)


        n_loss_terms = 0
        total_loss = 0
        for iq, q in enumerate(teacher_out):
            for v in range(len(student_out)):
                if v == iq:
                    continue


                loss = torch.sum(-q * F.log_softmax(student_out[v], dim=-1), dim=-1)
                total_loss += loss.mean()
                n_loss_terms += 1

        total_loss /= n_loss_terms

        total_loss = dict( ce_loss=total_loss, loss=total_loss)
        self.update_center(teacher_output)
        return total_loss



    @torch.no_grad()
    def update_center(self, teacher_output):
        """
        Update center used for teacher output.
        """
        batch_center = torch.sum(teacher_output, dim=0, keepdim=True)
        dist.all_reduce(batch_center)
        batch_center = batch_center / (len(teacher_output) * dist.get_world_size())

        # ema update
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)


In [ ]:
args = {
    "dataset": "CIFAR10",
    "image_size": 32,
    "patch_size": 4,
    "mlp_head_in": 192,
    "embed_dim": 192,
    "global_crops_scale": (0.7, 1.0),
    "local_crops_scale": (0.2, 0.5),
    "local_crops_number": 8,
    "out_dim": 1024,
    "batch_size_per_gpu": 256,
    "output_dir": "/content/gdrive/MyDrive/checkpoints",
    "datapath": "/path/to/cifar10/train/folder",
    "dist_url": "env://",
    "num_workers": 2,
    "use_bn_in_head": False,
    "norm_last_layer": False,
    "gpu":0,
    "warmup_teacher_temp": 0.04,
    "warmup_teacher_temp_epochs": 10,
    "warmup_epochs": 30,
    "epochs": 200,
    "min_lr": 1e-6,
    "optimizer": "adamw",
    "teacher_temp": 0.07,
    "use_fp16": False,
    "lr": 0.0001,
    "weight_decay": 0.04,
    "weight_decay_end": 0.4,
    "momentum_teacher": 0.996,
    "freeze_last_layer": 1,
    "clip_grad": 3.0,
    "in_channels": 3,
    "qkv_bias": True,
    "drop_rate": 0,
    "drop_path_rate": 0.1,
    "saveckp_freq": 10

}

print(args)

utils.fix_random_seeds(0)
init_distributed_mode(args)



{'dataset': 'CIFAR10', 'image_size': 32, 'patch_size': 4, 'mlp_head_in': 192, 'embed_dim': 192, 'global_crops_scale': (0.7, 1.0), 'local_crops_scale': (0.2, 0.5), 'local_crops_number': 8, 'out_dim': 1024, 'batch_size_per_gpu': 256, 'output_dir': '/content/gdrive/MyDrive/checkpoints', 'datapath': '/path/to/cifar10/train/folder', 'dist_url': 'env://', 'num_workers': 2, 'use_bn_in_head': False, 'norm_last_layer': False, 'gpu': 0, 'warmup_teacher_temp': 0.04, 'warmup_teacher_temp_epochs': 10, 'warmup_epochs': 30, 'epochs': 200, 'min_lr': 1e-06, 'optimizer': 'adamw', 'teacher_temp': 0.07, 'use_fp16': False, 'lr': 0.0001, 'weight_decay': 0.04, 'weight_decay_end': 0.4, 'momentum_teacher': 0.996, 'freeze_last_layer': 1, 'clip_grad': 3.0, 'in_channels': 3, 'qkv_bias': True, 'drop_rate': 0, 'drop_path_rate': 0.1, 'saveckp_freq': 10}
Will run the code on one GPU.
| distributed init (rank 0): env://


In [ ]:
# ============ preparing data ... ============


transform = DataAugmentation(
    args
)
if args["dataset"] == 'Tiny-Imagenet':
    dataset =  datasets.ImageFolder(
        root=args["datapath"], transform=transform)
elif args["dataset"] == "CIFAR10":
    dataset = torchvision.datasets.CIFAR10(root=args["datapath"], train=True,
                                    download=True, transform=transform)
elif args["dataset"] == "CIFAR100":
    dataset = torchvision.datasets.CIFAR100(root=args["datapath"], train=True,
                                    download=True, transform=transform)
elif args["dataset"] == "CINIC":
    dataset = datasets.ImageFolder(root=args["datapath"],transform=transform)

elif args["dataset"] == "SVHN":
    dataset = torchvision.datasets.SVHN(root=args["datapath"],split='train',
                                      download=True, transform=transform)

sampler = torch.utils.data.DistributedSampler(dataset, shuffle=True)
data_loader = torch.utils.data.DataLoader(
    dataset,
    sampler=sampler,
    batch_size=args["batch_size_per_gpu"],
    num_workers=args["num_workers"],
    pin_memory=True,
    drop_last=True,
)
print(f"Data loaded: there are {len(dataset)} images.")

100%|██████████| 170498071/170498071 [00:07<00:00, 21895262.15it/s]


Extracting /path/to/cifar10/train/folder/cifar-10-python.tar.gz to /path/to/cifar10/train/folder
Data loaded: there are 50000 images.


In [ ]:
# from vit_pytorch import ViT

device = "cuda"
print(args)


student = VisionTransformer(img_size=[args["image_size"]],
    patch_size=args["patch_size"],
    in_chans=args["in_channels"],
    num_classes=0,
    embed_dim=192,
    depth=9,
    num_heads=12,
    mlp_ratio=2,
    qkv_bias=args["qkv_bias"],
    drop_rate=args["drop_rate"],
    drop_path_rate=args["drop_path_rate"],
    norm_layer=partial(nn.LayerNorm, eps=1e-6))

teacher = VisionTransformer(img_size=[args["image_size"]],
    patch_size=args["patch_size"],
    in_chans=args["in_channels"],
    num_classes=0,
    embed_dim=192,
    depth=9,
    num_heads=12,
    mlp_ratio=2,
    qkv_bias=args["qkv_bias"],
    drop_rate=args["drop_rate"],
    drop_path_rate=args["drop_path_rate"],
    norm_layer=partial(nn.LayerNorm, eps=1e-6))


student = utils.MultiCropWrapper(student, projection_head.MLPHead(args["mlp_head_in"], args["out_dim"], args["use_bn_in_head"]))

teacher = utils.MultiCropWrapper(
    teacher, projection_head.MLPHead(
      args["mlp_head_in"],
      args["out_dim"],
      use_bn=args["use_bn_in_head"],
      norm_last_layer=args["norm_last_layer"]
)
)


{'dataset': 'CIFAR10', 'image_size': 32, 'patch_size': 4, 'mlp_head_in': 192, 'embed_dim': 192, 'global_crops_scale': (0.7, 1.0), 'local_crops_scale': (0.2, 0.5), 'local_crops_number': 8, 'out_dim': 1024, 'batch_size_per_gpu': 256, 'output_dir': '/content/gdrive/MyDrive/checkpoints', 'datapath': '/path/to/cifar10/train/folder', 'dist_url': 'env://', 'num_workers': 2, 'use_bn_in_head': False, 'norm_last_layer': False, 'gpu': 0, 'warmup_teacher_temp': 0.04, 'warmup_teacher_temp_epochs': 10, 'warmup_epochs': 30, 'epochs': 200, 'min_lr': 1e-06, 'optimizer': 'adamw', 'teacher_temp': 0.07, 'use_fp16': False, 'lr': 0.0001, 'weight_decay': 0.04, 'weight_decay_end': 0.4, 'momentum_teacher': 0.996, 'freeze_last_layer': 1, 'clip_grad': 3.0, 'in_channels': 3, 'qkv_bias': True, 'drop_rate': 0, 'drop_path_rate': 0.1, 'saveckp_freq': 10, 'rank': 0, 'world_size': 1}


/usr/local/lib/python3.10/dist-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [ ]:
# move networks to gpu
student, teacher = student.cuda(), teacher.cuda()

# synchronize batch norms (if any)
if utils.has_batchnorms(student):
    student = nn.SyncBatchNorm.convert_sync_batchnorm(student)
    teacher = nn.SyncBatchNorm.convert_sync_batchnorm(teacher)

    # we need DDP wrapper to have synchro batch norms working...
    teacher = nn.parallel.DistributedDataParallel(teacher, device_ids=[args["gpu"]])
    teacher_without_ddp = teacher.module
else:
    # teacher_without_ddp and teacher are the same thing
    teacher_without_ddp = teacher
student = nn.parallel.DistributedDataParallel(student, device_ids=[args["gpu"]])


# teacher and student start with the same weights
filtered_student_state_dict = {k: v for k, v in student.module.state_dict().items() if k in teacher_without_ddp.state_dict().keys()}
teacher_without_ddp.load_state_dict(filtered_student_state_dict)

# there is no backpropagation through the teacher, so no need for gradients
for p in teacher.parameters():
    p.requires_grad = False

print(f"Student and Teacher are built: they are both vit network.")


Student and Teacher are built: they are both vit network.


In [ ]:
# ============ preparing loss ... ============
view_pred_loss = ViewPredLoss(
    args["out_dim"],
    args["local_crops_number"] + 2,  # total number of crops = 2 global crops + local_crops_number
    args["warmup_teacher_temp"],
    args["teacher_temp"],
    args["warmup_teacher_temp_epochs"],
    args["epochs"],
).cuda()

# ============ preparing optimizer ... ============
params_groups = utils.get_params_groups(student)
if args["optimizer"] == "adamw":
    optimizer = torch.optim.AdamW(params_groups)  # to use with ViTs
elif args["optimizer"] == "sgd":
    optimizer = torch.optim.SGD(params_groups, lr=0, momentum=0.9)  # lr is set by scheduler
elif args["optimizer"] == "lars":
    optimizer = utils.LARS(params_groups)  # to use with convnet and large batches
# for mixed precision training
fp16_scaler = None
if args["use_fp16"]:
    fp16_scaler = torch.cuda.amp.GradScaler()

# ============ init schedulers ... ============
lr_schedule = utils.cosine_scheduler(
    args["lr"] * (args["batch_size_per_gpu"] * utils.get_world_size()) / 256. ,  # linear scaling rule
    args["min_lr"],
    args["epochs"], len(data_loader),
    warmup_epochs=args["warmup_epochs"],
)
wd_schedule = utils.cosine_scheduler(
    args["weight_decay"],
    args["weight_decay_end"],
    args["epochs"], len(data_loader),
)
# momentum parameter is increased to 1. during training with a cosine schedule
momentum_schedule = utils.cosine_scheduler(args["momentum_teacher"], 1,
                                            args["epochs"], len(data_loader))
print(f"Loss, optimizer and schedulers ready.")

Loss, optimizer and schedulers ready.


In [ ]:
def train_one_epoch(student, teacher, teacher_without_ddp, view_pred_loss, data_loader,
                    optimizer, lr_schedule, wd_schedule, momentum_schedule,epoch,
                    fp16_scaler, args):
    metric_logger = utils.MetricLogger(delimiter="  ")
    header = 'Epoch: [{}/{}]'.format(epoch, args["epochs"])

    for it, (images,_) in enumerate(metric_logger.log_every(data_loader, 10, header)):   #add mask here
        # update weight decay and learning rate according to their schedule
        it = len(data_loader) * epoch + it  # global training iteration
        for i, param_group in enumerate(optimizer.param_groups):
            param_group["lr"] = lr_schedule[it]
            if i == 0:  # only the first group is regularized
                param_group["weight_decay"] = wd_schedule[it]

        # move images to gpu
        images = [im.cuda(non_blocking=True) for im in images]



        # teacher and student forward passes + compute loss
        with torch.cuda.amp.autocast(fp16_scaler is not None):
            teacher_output  = teacher(images[:2])  # only the 2 global views pass through the teacher
            student_output  = student(images)


            total_loss = view_pred_loss(student_output, teacher_output, epoch)
            loss = total_loss.pop('loss')
            loss_view = total_loss.pop('ce_loss')

        if not math.isfinite(loss.item()):
            print("Loss is {}, View Pred loss is {}, stopping training".format(loss.item(),loss_view.item()
                                                                                           ), force=True)
            sys.exit(1)


        # student update
        optimizer.zero_grad()
        param_norms = None
        if fp16_scaler is None:
            loss.backward()
            if args["clip_grad"]:
                param_norms = utils.clip_gradients(student, args["clip_grad"])
            utils.cancel_gradients_last_layer(epoch, student,
                                              args["freeze_last_layer"])
            optimizer.step()
        else:
            fp16_scaler.scale(loss).backward()
            if args["clip_grad"]:
                fp16_scaler.unscale_(optimizer)  # unscale the gradients of optimizer's assigned params in-place
                param_norms = utils.clip_gradients(student, args["clip_grad"])
            utils.cancel_gradients_last_layer(epoch, student,
                                              args["freeze_last_layer"])
            fp16_scaler.step(optimizer)
            fp16_scaler.update()


        # EMA update for the teacher
        with torch.no_grad():
            m = momentum_schedule[it]  # momentum parameter

            names_q, params_q, names_k, params_k = [], [], [], []
            for name_q, param_q in student.module.named_parameters():
                names_q.append(name_q)
                params_q.append(param_q)
            for name_k, param_k in teacher_without_ddp.named_parameters():
                names_k.append(name_k)
                params_k.append(param_k)
            names_common = list(set(names_q) & set(names_k))
            params_q = [param_q for name_q, param_q in zip(names_q, params_q) if name_q in names_common]
            params_k = [param_k for name_k, param_k in zip(names_k, params_k) if name_k in names_common]

            for param_q, param_k in zip(params_q, params_k):
                param_k.data.mul_(m).add_((1 - m) * param_q.detach().data)



            # for param_q, param_k in zip(student.module.parameters(), teacher_without_ddp.parameters()):
            #     param_k.data.mul_(m).add_((1 - m) * param_q.detach().data)

        # logging
        torch.cuda.synchronize()
        metric_logger.update(loss=loss.item())
        metric_logger.update(view_pred_loss=loss_view.item())
        metric_logger.update(lr=optimizer.param_groups[0]["lr"])
        metric_logger.update(wd=optimizer.param_groups[0]["weight_decay"])
    # gather the stats from all processes
    print("loop end")
    metric_logger.synchronize_between_processes()
    print("Averaged stats:", metric_logger)
    return {k: meter.global_avg for k, meter in metric_logger.meters.items()}

In [ ]:
# Make sure the output directory exists
# output_dir = args["output_dir"]
# os.makedirs(output_dir, exist_ok=True)

# import shutil
# # Or, remove the directory and its contents recursively
# shutil.rmtree(output_dir)



In [ ]:


# ============ optionally resume training ... ============
to_restore = {"epoch": 0}
utils.restart_from_checkpoint(
    os.path.join(args["output_dir"], "checkpoint.pth"),
    run_variables=to_restore,
    student=student,
    teacher=teacher,
    optimizer=optimizer,
    fp16_scaler=fp16_scaler,
    view_pred_loss=view_pred_loss,
)
start_epoch = to_restore["epoch"]

start_time = time.time()
print("Starting SSL training !")
for epoch in range(start_epoch, args["epochs"]):


    data_loader.sampler.set_epoch(epoch)

    # ============ training one epoch ... ============
    train_stats = train_one_epoch(student, teacher, teacher_without_ddp, view_pred_loss,
        data_loader, optimizer, lr_schedule, wd_schedule, momentum_schedule,
        epoch, fp16_scaler, args)

    # ============ writing logs ... ============
    save_dict = {
        'student': student.state_dict(),
        'teacher': teacher.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch + 1,
        'args': args,
        'view_pred_loss': view_pred_loss.state_dict(),
    }
    if fp16_scaler is not None:
        save_dict['fp16_scaler'] = fp16_scaler.state_dict()
    utils.save_on_master(save_dict, os.path.join(args["output_dir"], 'checkpoint.pth'))
    if args["saveckp_freq"] and epoch % args["saveckp_freq"] == 0:
        utils.save_on_master(save_dict, os.path.join(args["output_dir"], f'checkpoint{epoch:04}.pth'))
    log_stats = {**{f'train_{k}': v for k, v in train_stats.items()},
                  'epoch': epoch}
    if utils.is_main_process():
        with (Path(args["output_dir"]) / "log.txt").open("a") as f:
            f.write(json.dumps(log_stats) + "\n")
total_time = time.time() - start_time
total_time_str = str(datetime.timedelta(seconds=int(total_time)))
print('Training time {}'.format(total_time_str))

Found checkpoint at /content/gdrive/MyDrive/checkpoints/checkpoint.pth
=> loaded 'student' from checkpoint '/content/gdrive/MyDrive/checkpoints/checkpoint.pth' with msg <All keys matched successfully>
=> loaded 'teacher' from checkpoint '/content/gdrive/MyDrive/checkpoints/checkpoint.pth' with msg <All keys matched successfully>
=> loaded 'optimizer' from checkpoint: '/content/gdrive/MyDrive/checkpoints/checkpoint.pth'
=> key 'fp16_scaler' not found in checkpoint: '/content/gdrive/MyDrive/checkpoints/checkpoint.pth'
=> loaded 'view_pred_loss' from checkpoint '/content/gdrive/MyDrive/checkpoints/checkpoint.pth' with msg <All keys matched successfully>
Starting SSL training !
Epoch: [180/200]  [  0/195]  eta: 0:39:49  loss: 4.756777 (4.756777)  view_pred_loss: 4.756777 (4.756777)  lr: 0.000004 (0.000004)  wd: 0.391190 (0.391190)  time: 12.256281  data: 6.938762  max mem: 6864
Epoch: [180/200]  [ 10/195]  eta: 0:09:55  loss: 4.756777 (4.761623)  view_pred_loss: 4.756777 (4.761623)  lr: 0.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torchvision==0.14.0

In [ ]:
import torchvision

In [ ]:
import torchvision.transforms as transform
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import random_split


transform = transform.Compose([
    transform.ToTensor(),  # Convert the image to a PyTorch tensor
    transform.RandomCrop(32, padding=4),  # Random crop with padding
    transform.RandomHorizontalFlip(),     # Random horizontal flip
    transform.Normalize(mean=data_info['stat'][0], std=data_info['stat'][1])
    # Add more transformations as needed
])
train_dataset = torchvision["dataset"].CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision["dataset"].CIFAR10(root='./data', train=False, download=True, transform=transform)
validation_size = int(0.2 * len(train_dataset))
train_size = len(train_dataset) - validation_size
train_dataset, validation_dataset = random_split(train_dataset, [train_size, validation_size])
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
import torch
from vit_pytorch import ViT

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:

student = ViT(
    image_size = 32,
    patch_size = 4,
    num_classes = 0,
    dim = 192,
    depth = 9,     # list of depths, indicating the number of rounds of each stage before a downsample
    heads = 12,
    mlp_dim = 192,
    dropout = 0.2,
    emb_dropout = 0.1
).to(device)


teacher = ViT(
    image_size = 32,
    patch_size = 4,
    num_classes = 0,
    dim = 192,
    depth = 9,     # list of depths, indicating the number of rounds of each stage before a downsample
    heads = 12,
    mlp_dim = 192,
    dropout = 0.2,
    emb_dropout = 0.1
).to(device)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
lr = 0.001
gamma = 0.5
epochs =100
# loss function
criterion = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
# scheduler
scheduler = StepLR(optimizer, step_size=5, gamma=gamma)

In [ ]:
import matplotlib.pyplot as plt

# Lists to store training and validation metrics
train_loss_list = []
train_accuracy_list = []
val_loss_list = []
val_accuracy_list = []

In [ ]:
import os
from tqdm.notebook import tqdm
# Define the path for saving checkpoints
# checkpoint_path = 'model_checkpoint.pth'

# Optionally, load a saved checkpoint to resume training
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     epoch_start = checkpoint['epoch'] + 1
#     best_val_loss = checkpoint['best_val_loss']
# else:
#     epoch_start = 0
#     best_val_loss = float('inf')

# Training loop
for epoch in range(1, epochs+1):
    epoch_loss = 0
    epoch_accuracy = 0

    for data, label in tqdm(train_loader):
        data = data.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, label)


        loss.backward()
        optimizer.step()

        acc = (output.argmax(dim=1) == label).float().mean()
        epoch_accuracy += acc / len(train_loader)
        epoch_loss += loss / len(train_loader)

    #Validation
    model.eval()
    with torch.no_grad():
        epoch_val_accuracy = 0
        epoch_val_loss = 0
        for data, label in validation_loader:
            data = data.to(device)
            label = label.to(device)

            val_output = model(data)
            val_loss = criterion(val_output, label)

            acc = (val_output.argmax(dim=1) == label).float().mean()
            epoch_val_accuracy += acc / len(validation_loader)
            epoch_val_loss += val_loss / len(validation_loader)
    scheduler.step()

    train_loss_list.append(epoch_loss.item())
    train_accuracy_list.append(epoch_accuracy.item())
    val_loss_list.append(epoch_val_loss.item())
    val_accuracy_list.append(epoch_val_accuracy.item())

    print(
        f"Epoch : {epoch} - loss : {epoch_loss:.4f} - acc: {epoch_accuracy:.4f} - val_loss : {epoch_val_loss:.4f} - val_acc: {epoch_val_accuracy:.4f}\n"
    )

    # Save checkpoint at the end of each epoch
#     torch.save({
#         'epoch': epoch,
#         'model_state_dict': model.state_dict(),
#         'optimizer_state_dict': optimizer.state_dict(),
#         'best_val_loss': best_val_loss
#     }, checkpoint_path)

In [ ]:
plt.figure(figsize=(12, 5))

# Plotting Loss Curves
plt.subplot(1, 2, 1)
plt.plot(train_loss_list, label='Training Loss', marker='o')
plt.plot(val_loss_list, label='Validation Loss', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curves')
plt.legend()
# Plotting Accuracy Curves
plt.subplot(1, 2, 2)
plt.plot(train_accuracy_list, label='Training Accuracy', marker='o')
plt.plot(val_accuracy_list, label='Validation Accuracy', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy Curves')
plt.legend()

plt.tight_layout()
plt.show()

